In [ ]:
%matplotlib ipympl

In [ ]:
from __future__ import annotations

import dataclasses
import functools
import typing as tp
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as sci_interp
import scipy.optimize as sci_opt
import scipy.signal as sci_sig
from flax import nnx

jax.config.update("jax_enable_x64", True)

In [ ]:
def load_clean_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    data = np.array(pd.read_hdf(file_path))
    return data[:, 1:4], data[:, 4:]

file_path = "/Users/jozbee/work/eng/comp/data/clean_specific-forces-standard-road-v2.hdf"
test_file_path = "/Users/jozbee/work/eng/comp/data/clean_00_sms_drive.hdf"
acc_ref, omega_ref = load_clean_references(file_path)
test_acc_ref, test_omega_ref = load_clean_references(test_file_path)

acc_ref = jnp.clip(acc_ref, -1.0, 1.0)
test_acc_ref = jnp.clip(test_acc_ref, -1.0, 1.0)

In [ ]:
def smooth_data(data, nu=0):
    ts = np.arange(data.size) * dt
    return sci_interp.make_smoothing_spline(ts, data, lam=1e0)(ts, nu=nu)

In [ ]:
# data_range = [0, 90 * 200]
# data_range = [0, 4000]
# data_range = [8000, 12000]
# data_range = [8000, 90 * 200]
# data_range = [8000, 150 * 200]
data_range = [0, 250 * 200]
# data_range = [0, 2000]

dt = 0.005
ts = np.arange(*data_range) * dt
data = acc_ref[data_range[0]: data_range[1], 0]
# data = test_acc_ref[data_range[0]: data_range[1], 0]
ref_data = smooth_data(data)
ref_datap = smooth_data(data, nu=1)

## filt

In [ ]:
def fast_trip_E0(f: jax.Array) -> jax.Array:
    x0 = f**2
    x1 = x0 + 80000
    x2 = jnp.exp((1/200)*f)
    x3 = (1/80000)*x2
    x4 = (1/40000)*x2
    x5 = f**3
    x6 = x3*(f + 400)
    return jnp.array([[x3*(800*f + x1), x0*x4*(-f - 600), x5*x6], [x6, x4*(-200*f - x0 + 40000), x3*x5], [x3, x4*(200 - f), x3*(-400*f + x1)]])

def fast_trip_E1(f: jax.Array) -> jax.Array:
    x0 = (1/200)*f
    x1 = jnp.exp(x0)
    x2 = (1/80000)*x1
    return jnp.array([[x2*(f + 400)], [x2], [(f**2*x2 - x0*x1 + x1 - 1)/f**3]])

def fast_trip_C(f: jax.Array, nu: int) -> jax.Array:
    assert 0 <= nu and nu <= 2
    if nu == 0:
        return jnp.array([0, 0, (-f)**3])
    elif nu == 1:
        return jnp.array([0, (-f)**3, 0])
    else:  # nu == 2
        return jnp.array([(-f)**3, 0, 0])

@functools.partial(jax.jit, static_argnames=["nu"])
def fast_trip_E0_E1_C(f: jax.Array, nu: int=0) -> tuple[jax.Array, jax.Array, jax.Array]:
    return fast_trip_E0(f), jnp.ravel(fast_trip_E1(f)), fast_trip_C(f, nu)

@jax.jit
def fast_obs_x0(f, y_0, y_1, y_2, u_0, u_1):
    x0 = f**3
    x1 = x0**(-1.0)
    x2 = f*y_2
    x3 = f**2
    x4 = jnp.exp((1/200)*f)
    x5 = jnp.exp((1/100)*f)
    x6 = x5*y_0
    x7 = 160000*x4
    x8 = f*u_0
    x9 = x3*x4
    return jnp.ravel(jnp.array([[(1/400)*x1*(80000*f*u_0*x5 - f*u_1*x7 + 240000*f*u_1 + 320000*f*x4*y_1 - 80000*f*x6 - u_0*x0*x4 - 16000000*u_0*x4 + 16000000*u_0*x5 - 600*u_0*x9 + u_1*x0*x4 + 600*u_1*x3*x4 + 400*u_1*x3 - 16000000*u_1*x4 + 16000000*u_1 - 240000*x2 - 400*x3*y_2 + 32000000*x4*y_1 - 16000000*x6 - x7*x8 - 16000000*y_2)], [x1*((1/2)*f*u_1*x4 + f*u_1 - 100*u_0*x4 + 100*u_0*x5 - 1/800*u_0*x9 + (1/800)*u_1*x3*x4 - 300*u_1*x4 + 300*u_1 - x2 - 1/2*x4*x8 + 400*x4*y_1 - 100*x6 - 300*y_2)], [-x1*y_2]]))


In [ ]:
heur_size = 50

@functools.partial(jax.jit, static_argnames=["nu"])
def cutoff_linear_filt(
    fast_cutoff: jax.Array,
    slow_cutoff: jax.Array,
    data: jax.Array,
    nu: int=0,
) -> tuple[jax.Array, jax.Array]:
    y = jnp.zeros(data.size + heur_size)
    fs = jnp.zeros(data.size)
    up = jnp.concatenate([jnp.ones(heur_size) * data[0], data])  # data (u) padded

    def filt_body(i: int, state: tuple[jax.Array, jax.Array]) -> tuple[jax.Array, jax.Array]:
        y, fs = state
        u_hist = jax.lax.dynamic_slice(up, [i], [heur_size])
        y_hist = jax.lax.dynamic_slice(y, [i], [heur_size])
        diff = u_hist - y_hist
        heur = jnp.abs(jnp.mean(diff))
        abs_heur = jnp.mean(jnp.abs(diff))
        res_heur = abs_heur - heur
        f = jax.lax.cond(
            (heur > 0.2) | (res_heur < 0.01),
            lambda: fast_cutoff,
            lambda: slow_cutoff,
        )
        fs = fs.at[i].set(f)
        E0, E1, C = fast_trip_E0_E1_C(f, nu)

        idx = i + heur_size  # index past history
        u = up[idx]  # not included in `u_hist`
        x0 = fast_obs_x0(f, y[idx - 3], y[idx - 2], y[idx - 1], up[idx - 1], up[idx - 2])
        x1 = E0 @ x0 + E1 * u
        yi = C @ x1
        y = y.at[idx].set(yi)
        return y, fs

    y, fs = jax.lax.fori_loop(0, data.size, filt_body, (y, fs))
    y = y[heur_size:]  # remove initial padding
    return y, fs

## analysis

In [ ]:
sms_cutoffs = (-2 * jnp.pi, -2 * jnp.pi)
# sms_cutoffs = (-10.0, -10.0)
# sms_cutoffs = (-5.0, -5.0)
heur_cutoffs = (-15.0, -5.0)

In [ ]:
interp_filt = sci_sig.medfilt(data, kernel_size=101)
ts = np.arange(data.size) * dt
gap = 100
# interp_filt = sci_interp.PchipInterpolator(ts[::gap], interp_filt[::gap])(ts)
# interp_filt = sci_interp.CubicSpline(ts[::gap], interp_filt[::gap])(ts)

In [ ]:
heuristic_filt, heuristic_fs = cutoff_linear_filt(*heur_cutoffs, data=data)
ref_filt, ref_fs = cutoff_linear_filt(*sms_cutoffs, data=data)

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(data, label="data", alpha=0.2)
# ax.plot(ref_data, label="ref_data", alpha=0.4)
ax.plot(heuristic_filt, label="heuristic_filt")
ax.plot(ref_filt, label="ref_filt")
# ax.plot(interp_filt, label="interp_filt")

# ax.set_ylim(-1, 1)
ax.legend()
ax.grid()

In [ ]:
# signed_diff = data - ref_filt
signed_diff = data - heuristic_filt
tmp_heur_size = heur_size
heur = np.abs(np.convolve(signed_diff, np.ones(tmp_heur_size) / tmp_heur_size, mode="valid"))
abs_heur = np.convolve(np.abs(signed_diff), np.ones(tmp_heur_size) / tmp_heur_size, mode="valid")
res_heur = abs_heur - heur  # residual heuristics

# mid = jnp.mean(jnp.array(heur_cutoffs))
# fast_slow = jnp.where(heuristic_fs < mid, 1.0, 0.8)
fast_slow = jnp.where((heur > 0.2) | (res_heur < 0.01), 1.0, 0.8)

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.plot(heur, label="heur")
ax.plot(abs_heur, label="abs_heur")
ax.plot(res_heur, label="res_heur")
ax.scatter(jnp.arange(fast_slow.size), fast_slow, label="fast_slow", marker="o", c="orange", s=2)
# yticks = np.arange(-13, 13) * 0.1
# ylabels = [f'{y:1.1f}' for y in yticks]
# ax.set_yticks(yticks, labels=ylabels)
ax.grid()
ax.legend()